In [1]:
import torch
import torch.nn as nn

In [2]:
# 1. THE ENCODER
encoder = nn.LSTM(input_size=10, hidden_size=20, batch_first=True)

In [3]:
# 2. THE DECODER (Usually has its own input size, e.g., target language)
decoder = nn.LSTM(input_size=10, hidden_size=20, batch_first=True)

In [5]:
# --- THE FLOW ---
spanish_sentence = torch.randn(1, 5, 10) # 1 sentence, 5 words

# Encoder processes the whole thing
_, (hn, cn) = encoder(spanish_sentence)

# The "thought vector" (hn, cn) becomes the starting point for the Decoder
# We give the decoder a "Start" token to begin
start_token = torch.randn(1, 1, 10) 

# Decoder generates the first English word's representation
decoder_output, (hn, cn) = decoder(start_token, (hn, cn))

print(f"Decoder output shape: {decoder_output.shape}")

Decoder output shape: torch.Size([1, 1, 20])


# Exercise
The Setup:

Encoder: Takes Python tokens (input_size = 100) and creates a "thought vector" (hidden_size = 256).

Decoder: Takes that "thought vector" and starts generating English words (input_size = 100).

In [ ]:
import torch
import torch.nn as nn

class Seq2Seq(nn.Module):
    def __init__(self):
        super(Seq2Seq, self).__init__()
        # Encoder: Reads Python
        self.encoder = nn.LSTM(input_size=100, hidden_size=256, batch_first=True)
        
        # Decoder: Writes English
        self.decoder = nn.LSTM(input_size=100, hidden_size=256, batch_first=True)
        
    def forward(self, source_code, target_start_token):
        # 1. Pass the source_code through the encoder
        # We don't need the sequence of outputs, just the final states
        _, (hn, cn) = self.encoder(source_code)
        
        # 2. Pass the target_start_token through the decoder
        # CRITICAL: The decoder MUST start with the encoder's final (hn, cn)
        decoder_output, (hn_dec, cn_dec) = self.decoder(target_start_token, (hn, cn))
        
        return decoder_output

# Test setup
model = Seq2Seq()
py_code = torch.randn(1, 10, 100) # 1 snippet, 10 tokens
start_token = torch.randn(1, 1, 100) # The "start" signal for English

output = model(py_code, start_token)
print(f"Output shape: {output.shape}") # Should be (1, 1, 256)